In [1]:


# ================================
# IMPORTS
# ================================
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq


# ================================
# CONFIG
# ================================
GROQ_API_KEY = "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"

faiss_index_path = "data/vector_store/faiss_index.bin"
metadata_path = "data/vector_store/metadata.json"


# ================================
# LOAD DATA
# ================================
index = faiss.read_index(faiss_index_path)

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
client = Groq(api_key=GROQ_API_KEY)


# ================================
# RETRIEVAL
# ================================
def retrieve(query, top_k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for i, idx in enumerate(indices[0]):
        if idx >= len(metadata):
            continue

        item = metadata[idx]

        results.append({
            "text": item["text"],
            "source": item["source"],
            "domain": item["domain"],
            "score": float(distances[0][i])
        })

    return results


# ================================
# CONTEXT BUILDER
# ================================
def build_context(chunks):
    context = ""
    for i, chunk in enumerate(chunks):
        context += f"\n[Chunk {i+1}]\n{chunk['text']}\n"
    return context


# ================================
# GENERATE ANSWER
# ================================
def generate_answer(query):
    chunks = retrieve(query)

    context = build_context(chunks)

    prompt = f"""
You are KrishiSamadhan, an agricultural assistant.

Answer ONLY from context.
If not found, say "I don't have enough information."

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=300
        )

        answer = response.choices[0].message.content

    except Exception as e:
        print("LLM Error:", e)
        answer = "Error generating answer."

    return answer, chunks


# ================================
# ADVANCED METRIC
# Context Relevance Score
# ================================
def context_relevance(query, chunks):
    query_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_emb)

    scores = []

    for chunk in chunks:
        chunk_emb = model.encode([chunk["text"]], convert_to_numpy=True)
        faiss.normalize_L2(chunk_emb)

        score = np.dot(query_emb, chunk_emb.T)[0][0]
        scores.append(score)

    return float(np.mean(scores)) if scores else 0.0


# ================================
# TEST QUERIES
# ================================
test_queries = [
    "best fertilizer for wheat crop",
    "how to control pests in rice",
    "how to improve soil fertility",
    "irrigation methods for farming"
]


# ================================
# RUN EVALUATION
# ================================
results_summary = []

print("\n🔹 Running Evaluation...\n")

for query in test_queries:
    print(f"\n📌 Query: {query}")

    answer, chunks = generate_answer(query)

    relevance_score = context_relevance(query, chunks)

    print("\nAnswer:")
    print(answer)

    print("\nRelevance Score:", round(relevance_score, 4))

    print("\nSources:")
    for c in chunks:
        print("-", c["source"])

    print("\n=============================\n")

    results_summary.append({
        "query": query,
        "answer": answer,
        "relevance_score": relevance_score
    })


# ================================
# FINAL SUMMARY
# ================================
avg_score = np.mean([r["relevance_score"] for r in results_summary])

print("\n📊 FINAL METRICS")
print(f"Average Context Relevance Score: {avg_score:.4f}")

e:\Udemy ML course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3171.62it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🔹 Running Evaluation...


📌 Query: best fertilizer for wheat crop

Answer:
Based on the context provided, the best fertilizer for a wheat crop would likely be a compound fertilizer with an NPK ratio of 10-20-10. This ratio is suitable for most plants and soil types, as mentioned in Chunk 2. However, if the soil type is sandy or clay, a higher NPK level, such as 15-30-15, may be required.

It's also worth noting that wheat is a cereal crop, and cereal production accounts for about 50% of world fertilizer use, as mentioned in Chunk 4. Therefore, a fertilizer that is specifically formulated for cereal crops may be a good option.

In terms of specific fertilizer types, ammonium nitrate (AN) and calcium ammonium nitrate (CAN) are two popular types of nitrogen-based fertilizers that are commonly used for cereal crops, as mentioned in Chunk 3. However, the best fertilizer for a wheat crop will ultimately depend on the specific soil type, climate, and management practices used in the region.
